# AIS 피처 ablation — HDG 기여도 정량 검증

수중 음향 선박 인식 / **Task 1 선종 4-class (Macro F1)**

두 실험을 한 노트북에서 수행한다.

- **PART 1 — 음향 융합 ablation**: EfficientNet-B0 + AIS late-fusion
  - 모델 A (4-d): `[sog_norm, cog_sin, cog_cos, sog_zero]` (HDG 제외)
  - 모델 B (7-d): `[sog_norm, cog_sin, cog_cos, hdg_sin, hdg_cos, cog_missing, hdg_missing]`
- **PART 2 — AIS-only LightGBM ablation**: 융합에서 묻히는 HDG 순수 기여도
  - 모델 C (4-d, A 인코딩) / 모델 D (7-d, B 인코딩) / 모델 Dp = D' (raw, LGBM NaN 처리)

> ⚠️ **실행 환경**: PART 1은 GPU 필수(EfficientNet-B0 학습). GPU 서버에서 실행할 것.
> 필요 패키지: torch, timm, librosa, lightgbm, shap(선택), pandas, matplotlib, scikit-learn.
> **mel 캐시**: 기본은 서버의 기존 `data/spec_cache`(train과 동일, repo 파이프라인 fmax=16000) 재사용. 셀 2의 `REUSE_EXISTING_CACHE` 토글 참고.

In [ ]:
import os, time, math, random, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    except Exception:
        pass
seed_everything(SEED)

# ---- 경로 (EDA와 동일: notebooks/ 에서 열면 ../data, 루트에서 열면 data) ----
BASE = '../data' if os.path.isdir('../data/train') else 'data'
TRAIN_CSV   = BASE + '/train/train.csv'
TRAIN_AUDIO = BASE + '/train/audio'
OOF_DIR     = 'oof'; os.makedirs(OOF_DIR, exist_ok=True)

CLASSES = ['A_SmallWorking', 'B_MotorBoat', 'C_Passenger', 'D_LargeShip']
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}

# ---- 음향 파라미터 ----
SR=32000; DURATION=5.0; N_MELS=128; N_FFT=2048; HOP=512; FMIN=50

# ---- mel 캐시 선택 ----
# 서버 data/spec_cache 에 train 캐시가 이미 존재(repo 파이프라인 산출: fmax=16000, [0,1] 정규화).
# 본 노트북은 클립별 z-score 정규화를 적용 -> [0,1] vs dB(아핀 변환) 차이는 결과에 무해.
# 단 fmax 는 캐시 생성값과 반드시 일치시켜야 함(서로 다른 fmax npy 혼용 금지).
REUSE_EXISTING_CACHE = True
if REUSE_EXISTING_CACHE:
    CACHE_DIR = BASE + '/spec_cache'        # 기존 train 캐시 그대로 재사용 (재변환 없음)
    FMAX = 16000                            # 기존 캐시 파라미터에 맞춤
else:
    CACHE_DIR = BASE + '/spec_cache_f8000'  # 신규 전용 캐시
    FMAX = 8000                             # 설계서 사양. 첫 실행 시 35k 재변환(수십분)

# ---- 학습 하이퍼파라미터 (A/B 공통. 변경 가능 지점은 인코딩과 d_in 뿐) ----
N_FOLDS=5; EPOCHS=15; BATCH=32; LR=1e-3; WD=1e-4; NUM_WORKERS=4
TIME_MASK=20; FREQ_MASK=20; MIXUP_ALPHA=0.4
EMBED_AC=512; AIS_HID=64; AIS_OUT=32; FUSION_HID=128
AUG_SEED=12345
FOLDS_ACOUSTIC=[0]   # ablation 기본 fold0. 시간되면 [0,1,2]. 전체 OOF는 list(range(5))

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, '| torch', torch.__version__)
print('CACHE_DIR =', CACHE_DIR, '| FMAX =', FMAX, '| REUSE =', REUSE_EXISTING_CACHE)
if DEVICE.type == 'cpu':
    print('경고: GPU 미감지. PART1(음향)은 매우 느리다. GPU 서버에서 실행할 것.')

## 1. 데이터 로드 · 센티넬 검증 · fold 분할

In [ ]:
df = pd.read_csv(TRAIN_CSV)
print('shape:', df.shape, '| 컬럼:', list(df.columns))

y = df['ship_type'].map(CLS2IDX).values
groups = df['ship_id'].values
N = len(df)
sog = df['sog'].astype(float).values
cog = df['cog'].astype(float).values
hdg = df['true_heading'].astype(float).values

# 센티넬/결측 재확인 (EDA 정합성)
print('SOG==0  :', int((sog==0).sum()),   '(%.2f%%)' % ((sog==0).mean()*100))
print('COG==360:', int((cog==360).sum()), '(%.2f%%)' % ((cog==360).mean()*100))
print('HDG==0  :', int((hdg==0).sum()),   '(%.2f%%)' % ((hdg==0).mean()*100))
print('HDG==511:', int((hdg==511).sum()), '| HDG>360:', int((hdg>360).sum()))

from sklearn.model_selection import StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(sgkf.split(df, y, groups))

print()
print('=== fold별 val 클래스 클립수 (분할 균형 확인) ===')
for fi, (tr, va) in enumerate(folds):
    cnt = np.bincount(y[va], minlength=4)
    line = '  '.join('%s=%d' % (CLASSES[k], cnt[k]) for k in range(4))
    print('fold%d  val=%6d   %s' % (fi, len(va), line))

## 2. AIS 인코딩 함수 (A/B 음향융합용, C/D/Dp AIS-only용)

In [ ]:
def sog_logclip(s):
    return np.log1p(np.clip(s, 0, 30))

def fit_sog_stats(idx):
    # SOG 표준화 통계는 train fold에서만 계산 (누수 방지)
    v = sog_logclip(sog[idx])
    return float(v.mean()), float(v.std() + 1e-6)

def encode_A(idx, stats):
    m, s = stats
    sn = (sog_logclip(sog[idx]) - m) / s
    sz = (sog[idx] == 0).astype(np.float32)
    r = np.deg2rad(cog[idx]); csin = np.sin(r); ccos = np.cos(r)
    mask = (sog[idx] == 0)               # 정지 시 cog 무의미 -> 0 마스킹
    csin[mask] = 0; ccos[mask] = 0
    return np.stack([sn, csin, ccos, sz], axis=1).astype(np.float32)

def encode_B(idx, stats):
    m, s = stats
    sn = (sog_logclip(sog[idx]) - m) / s
    rc = np.deg2rad(cog[idx]); rh = np.deg2rad(hdg[idx])
    csin = np.sin(rc); ccos = np.cos(rc); hsin = np.sin(rh); hcos = np.cos(rh)
    cmiss = (cog[idx] == 360); hmiss = (hdg[idx] == 0) | (hdg[idx] > 360)
    csin[cmiss] = 0; ccos[cmiss] = 0; hsin[hmiss] = 0; hcos[hmiss] = 0
    return np.stack([sn, csin, ccos, hsin, hcos,
                     cmiss.astype(np.float32), hmiss.astype(np.float32)], axis=1).astype(np.float32)

def encode_C(idx):
    sc = np.clip(sog[idx], 0, 30).astype(np.float32)
    sz = (sog[idx] == 0).astype(np.float32)
    r = np.deg2rad(cog[idx]); csin = np.sin(r); ccos = np.cos(r)
    mask = (sog[idx] == 0); csin[mask] = 0; ccos[mask] = 0
    return np.stack([sc, csin, ccos, sz], axis=1).astype(np.float32)

def encode_D(idx):
    sc = np.clip(sog[idx], 0, 30).astype(np.float32)
    rc = np.deg2rad(cog[idx]); rh = np.deg2rad(hdg[idx])
    csin = np.sin(rc); ccos = np.cos(rc); hsin = np.sin(rh); hcos = np.cos(rh)
    cmiss = (cog[idx] == 360); hmiss = (hdg[idx] == 0) | (hdg[idx] > 360)
    csin[cmiss] = 0; ccos[cmiss] = 0; hsin[hmiss] = 0; hcos[hmiss] = 0
    return np.stack([sc, csin, ccos, hsin, hcos,
                     cmiss.astype(np.float32), hmiss.astype(np.float32)], axis=1).astype(np.float32)

def encode_Dp(idx):
    sr_ = sog[idx].astype(np.float32).copy()
    cr_ = cog[idx].astype(np.float32).copy()
    hr_ = hdg[idx].astype(np.float32).copy()
    cr_[cog[idx] == 360] = np.nan
    hr_[(hdg[idx] == 0) | (hdg[idx] > 360)] = np.nan
    return np.stack([sr_, cr_, hr_], axis=1).astype(np.float32)

FEAT_C  = ['sog_clip', 'cog_sin', 'cog_cos', 'sog_zero']
FEAT_D  = ['sog_clip', 'cog_sin', 'cog_cos', 'hdg_sin', 'hdg_cos', 'cog_missing', 'hdg_missing']
FEAT_Dp = ['sog_raw', 'cog_raw', 'hdg_raw']
print('인코딩 함수 준비 완료')

## PART 1 — 음향 융합 ablation (A 4-d vs B 7-d)

### mel 캐시 확인
- `REUSE_EXISTING_CACHE=True`(기본): 서버의 기존 `data/spec_cache`(repo `src/data/mel.py` 산출, fmax=16000, [0,1] 정규화) 재사용 → **재변환 없음**.
- `False`: `spec_cache_f8000`에 fmax=8000으로 신규 빌드(35k, 수십분).

> A/B는 같은 캐시를 공유하므로 ablation(Δ_fusion) 결론은 fmax와 무관하게 유효.
> 절대 F1만 fmax 선택에 따라 달라진다.

In [ ]:
def wav_to_logmel(path):
    import librosa
    wav, _ = librosa.load(path, sr=SR, duration=DURATION, mono=True)
    tlen = int(SR * DURATION)
    if len(wav) < tlen:
        wav = np.pad(wav, (0, tlen - len(wav)))
    else:
        wav = wav[:tlen]
    mel = librosa.feature.melspectrogram(y=wav, sr=SR, n_mels=N_MELS, n_fft=N_FFT,
                                         hop_length=HOP, fmin=FMIN, fmax=FMAX)
    logmel = librosa.power_to_db(mel, ref=np.max)   # dB. 정규화는 로드시 클립별로.
    return logmel.astype(np.float32)

def build_cache(filenames, audio_dir, cache_dir):
    os.makedirs(cache_dir, exist_ok=True)
    todo = [f for f in filenames
            if not os.path.exists(os.path.join(cache_dir, f[:-4] + '.npy'))]
    print('cache:', len(filenames), '개 중 변환필요', len(todo), '->', cache_dir)
    if not todo:
        print('  -> 모두 존재. 재변환 없음.')
        return
    if REUSE_EXISTING_CACHE:
        # 재사용 모드인데 일부 누락: 다른 fmax로 누락분만 구우면 캐시가 섞임 -> 빌드 생략하고 경고만.
        print('  [중단] REUSE 모드에서 %d개 누락. 누락분을 fmax=%d로 새로 구우면 캐시가 혼용됨.' % (len(todo), FMAX))
        print('         기존 파이프라인으로 누락분을 채우거나, REUSE_EXISTING_CACHE=False 로 전체 재빌드할 것.')
        raise SystemExit('cache 누락 - 위 안내 참고')
    from tqdm import tqdm
    def one(f):
        try:
            np.save(os.path.join(cache_dir, f[:-4] + '.npy'),
                    wav_to_logmel(os.path.join(audio_dir, f)))
        except Exception as e:
            print('실패', f, e)
    try:
        from joblib import Parallel, delayed
        Parallel(n_jobs=-1)(delayed(one)(f) for f in tqdm(todo))
    except Exception:
        for f in tqdm(todo):
            one(f)

build_cache(df['filename'].tolist(), TRAIN_AUDIO, CACHE_DIR)

# 캐시 정합성 점검: shape(n_mels, T)와 값 범위 1개 확인
import glob
_some = sorted(glob.glob(os.path.join(CACHE_DIR, '*.npy')))[:1]
if _some:
    _s = np.load(_some[0])
    print('샘플 spec:', os.path.basename(_some[0]), '| shape=', _s.shape,
          '| min=%.3f max=%.3f' % (float(_s.min()), float(_s.max())))
    assert _s.shape[0] == N_MELS, 'n_mels 불일치: 캐시 %d vs 설정 %d' % (_s.shape[0], N_MELS)
    print('  n_mels=%d 일치. (z-score 정규화하므로 [0,1]이든 dB든 무해)' % N_MELS)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as Fn
from torch.utils.data import Dataset, DataLoader

class AisAudioDS(Dataset):
    def __init__(self, idx, ais):
        self.idx = np.asarray(idx)
        self.ais = ais.astype(np.float32)
        self.files = df['filename'].values
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        gi = self.idx[i]
        sp = np.load(os.path.join(CACHE_DIR, self.files[gi][:-4] + '.npy'))
        sp = (sp - sp.mean()) / (sp.std() + 1e-6)     # 클립별 정규화
        x = torch.from_numpy(sp).unsqueeze(0)
        a = torch.from_numpy(self.ais[i])
        return x, a, int(y[gi])

def make_loader(idx, ais, shuffle):
    g = torch.Generator(); g.manual_seed(SEED)        # 셔플 순서 고정 -> A/B 동일
    return DataLoader(AisAudioDS(idx, ais), batch_size=BATCH, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'),
                      generator=g, drop_last=False)

def spec_augment(x, gen):
    b_, _, M, T = x.shape
    x = x.clone()
    for b in range(b_):
        fl = int(torch.randint(1, FREQ_MASK + 1, (1,), generator=gen).item())
        f0 = int(torch.randint(0, max(1, M - fl), (1,), generator=gen).item())
        x[b, :, f0:f0 + fl, :] = 0
        tl = int(torch.randint(1, TIME_MASK + 1, (1,), generator=gen).item())
        t0 = int(torch.randint(0, max(1, T - tl), (1,), generator=gen).item())
        x[b, :, :, t0:t0 + tl] = 0
    return x

def mixup(x, lab, gen, nprng, alpha=MIXUP_ALPHA):
    lam = float(nprng.beta(alpha, alpha))
    perm = torch.randperm(x.size(0), generator=gen)
    xm = lam * x + (1 - lam) * x[perm]
    return xm, lab, lab[perm], lam

# augmentation 전용 RNG(gen/nprng)는 모델과 독립 -> d_in 차이가 aug를 바꾸지 않음(A/B 동일)
print('dataset / loader / augment 준비 완료')

In [ ]:
class FusionNet(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        import timm
        self.backbone = timm.create_model('tf_efficientnet_b0_ns', pretrained=True,
                                          num_classes=0, in_chans=1, global_pool='avg')
        bf = self.backbone.num_features
        self.ac = nn.Sequential(nn.Linear(bf, EMBED_AC), nn.BatchNorm1d(EMBED_AC), nn.GELU())
        self.fusion = nn.Sequential(nn.Linear(EMBED_AC + AIS_OUT, FUSION_HID), nn.ReLU(),
                                    nn.Dropout(0.3), nn.Linear(FUSION_HID, 4))
        # ais 브랜치를 마지막에 생성 -> 백본/ac/fusion init RNG를 A/B 동일하게 유지
        self.ais = nn.Sequential(nn.Linear(d_in, AIS_HID), nn.ReLU(), nn.Dropout(0.2),
                                 nn.Linear(AIS_HID, AIS_OUT))
    def forward(self, x, a):
        h = self.ac(self.backbone(x))
        z = self.ais(a)
        return self.fusion(torch.cat([h, z], dim=1))

from sklearn.metrics import f1_score

def warmup_cosine(step, total, warm):
    if step < warm:
        return (step + 1) / max(1, warm)
    p = (step - warm) / max(1, total - warm)
    return 0.5 * (1 + math.cos(math.pi * p))

def train_one_fold(d_in, tr_idx, va_idx, ais_tr, ais_va, class_w, tag):
    seed_everything(SEED)                  # 모델 init 직전 시드 고정
    model = FusionNet(d_in).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))
    cw = torch.tensor(class_w, dtype=torch.float32, device=DEVICE)
    tr_loader = make_loader(tr_idx, ais_tr, True)
    va_loader = make_loader(va_idx, ais_va, False)
    total = len(tr_loader) * EPOCHS; warm = len(tr_loader)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: warmup_cosine(s, total, warm))
    gen = torch.Generator(); nprng = np.random.RandomState()
    best = -1.0; best_ep = -1; best_pc = None; best_oof = None; hist = []
    t0 = time.time()
    for ep in range(EPOCHS):
        model.train()
        for bi, (x, a, lab) in enumerate(tr_loader):
            key = AUG_SEED + ep * 100000 + bi
            gen.manual_seed(key); nprng.seed((key + 777) % (2 ** 31))
            x = spec_augment(x, gen)
            x, ya, yb, lam = mixup(x, lab, gen, nprng)
            x = x.to(DEVICE); a = a.to(DEVICE); ya = ya.to(DEVICE); yb = yb.to(DEVICE)
            opt.zero_grad()
            with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                out = model(x, a)
                loss = (lam * Fn.cross_entropy(out, ya, weight=cw)
                        + (1 - lam) * Fn.cross_entropy(out, yb, weight=cw))
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
        model.eval(); P = []; YL = []
        with torch.no_grad():
            for x, a, lab in va_loader:
                x = x.to(DEVICE); a = a.to(DEVICE)
                with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                    out = model(x, a)
                P.append(torch.softmax(out, 1).float().cpu().numpy()); YL.append(lab.numpy())
        P = np.concatenate(P); YL = np.concatenate(YL)
        f1 = f1_score(YL, P.argmax(1), average='macro')
        pc = f1_score(YL, P.argmax(1), average=None, labels=[0, 1, 2, 3])
        hist.append(float(f1))
        if ep == 0:
            print('[%s] epoch1 sanity val MacroF1=%.4f' % (tag, f1))
        if f1 > best:
            best = f1; best_ep = ep + 1; best_pc = pc; best_oof = P.copy()
    mins = (time.time() - t0) / 60
    print('[%s] done best MacroF1=%.4f @ep%d  (%.1f분)' % (tag, best, best_ep, mins))
    return dict(hist=hist, best=float(best), best_ep=best_ep, pc=best_pc,
                oof=best_oof, va=va_idx, minutes=mins)

In [ ]:
oof_A = np.full((N, 4), np.nan, np.float32)
oof_B = np.full((N, 4), np.nan, np.float32)
res_ac = {'A': [], 'B': []}

for fi in FOLDS_ACOUSTIC:
    tr_idx, va_idx = folds[fi]
    stats = fit_sog_stats(tr_idx)                       # train fold SOG 통계 (A/B 공유)
    cnt = np.bincount(y[tr_idx], minlength=4).astype(np.float64)
    cw = 1.0 / cnt; cw = cw / cw.sum() * 4              # 1/클립수, 정규화(평균 1)
    print('--- fold%d | sog_stats=(%.3f, %.3f) | class_w=%s ---'
          % (fi, stats[0], stats[1], np.round(cw, 3)))
    rA = train_one_fold(4, tr_idx, va_idx, encode_A(tr_idx, stats), encode_A(va_idx, stats), cw, 'A.f%d' % fi)
    rB = train_one_fold(7, tr_idx, va_idx, encode_B(tr_idx, stats), encode_B(va_idx, stats), cw, 'B.f%d' % fi)
    rA['fold'] = fi; rB['fold'] = fi
    res_ac['A'].append(rA); res_ac['B'].append(rB)
    oof_A[va_idx] = rA['oof']; oof_B[va_idx] = rB['oof']

In [ ]:
import matplotlib.pyplot as plt

def agg(rs):
    best = float(np.mean([r['best'] for r in rs]))
    ep = int(round(np.mean([r['best_ep'] for r in rs])))
    pc = np.mean([r['pc'] for r in rs], axis=0)
    mins = float(np.sum([r['minutes'] for r in rs]))
    return best, ep, pc, mins

print('| 모델 | fold | best MacroF1 | best ep | F1_A | F1_B | F1_C | F1_D | 시간 |')
print('|------|------|--------------|---------|------|------|------|------|------|')
for tag in ['A', 'B']:
    dlabel = '4d' if tag == 'A' else '7d'
    for r in res_ac[tag]:
        pc = r['pc']
        print('| %s %s | %d | %.4f | %d | %.4f | %.4f | %.4f | %.4f | %.1f분 |'
              % (tag, dlabel, r['fold'], r['best'], r['best_ep'],
                 pc[0], pc[1], pc[2], pc[3], r['minutes']))

plt.figure(figsize=(8, 5))
for tag, col in [('A', '#4C72B0'), ('B', '#DD8452')]:
    if not res_ac[tag]:
        continue
    h = np.mean([r['hist'] for r in res_ac[tag]], axis=0)
    lab = 'Model A (4-d)' if tag == 'A' else 'Model B (7-d)'
    plt.plot(range(1, len(h) + 1), h, marker='o', color=col, label=lab)
plt.xlabel('epoch'); plt.ylabel('val Macro F1'); plt.title('Acoustic fusion: A vs B')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

pcA = agg(res_ac['A'])[2]; pcB = agg(res_ac['B'])[2]
d = pcB - pcA; k = int(np.argmax(np.abs(d)))
print('코멘트: 클래스 %s 에서 차이 최대 (B-A ΔF1=%+.4f).' % (CLASSES[k], d[k]))
print('클래스별 B-A ΔF1:', dict(zip(CLASSES, np.round(d, 4))))

## PART 2 — AIS-only LightGBM ablation (C / D / Dp=D')

In [ ]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

ALL = np.arange(N)
X    = {'C 4d': encode_C(ALL), 'D 7d': encode_D(ALL), 'Dp raw': encode_Dp(ALL)}
FEAT = {'C 4d': FEAT_C,        'D 7d': FEAT_D,        'Dp raw': FEAT_Dp}

# colsample_bytree=feature_fraction, subsample=bagging_fraction, subsample_freq=bagging_freq
LGB = dict(objective='multiclass', num_class=4, n_estimators=2000, learning_rate=0.05,
           num_leaves=31, min_child_samples=20, colsample_bytree=0.9, subsample=0.8,
           subsample_freq=5, class_weight='balanced', random_state=SEED, n_jobs=-1)

def run_lgbm(name):
    Xf = X[name]; oof = np.zeros((N, 4), np.float32)
    imp = np.zeros(Xf.shape[1]); last = None; t0 = time.time()
    for tr, va in folds:
        clf = LGBMClassifier(**LGB)
        clf.fit(Xf[tr], y[tr], eval_set=[(Xf[va], y[va])], eval_metric='multi_logloss',
                callbacks=[early_stopping(50), log_evaluation(0)])
        oof[va] = clf.predict_proba(Xf[va])
        imp += clf.booster_.feature_importance('gain'); last = clf
    f1 = f1_score(y, oof.argmax(1), average='macro')
    pc = f1_score(y, oof.argmax(1), average=None, labels=[0, 1, 2, 3])
    mins = (time.time() - t0) / 60
    print('[%s] OOF MacroF1=%.4f  (%.1f분)' % (name, f1, mins))
    return dict(name=name, oof=oof, f1=float(f1), pc=pc, imp=imp / N_FOLDS,
                feat=FEAT[name], minutes=mins, model=last)

rC = run_lgbm('C 4d'); rD = run_lgbm('D 7d'); rDp = run_lgbm('Dp raw')

print()
print('| 모델 | OOF MacroF1 | F1_A | F1_B | F1_C | F1_D | 시간 |')
print('|--------|-------------|------|------|------|------|------|')
for r in [rC, rD, rDp]:
    pc = r['pc']
    print('| %-6s | %.4f | %.4f | %.4f | %.4f | %.4f | %.1f분 |'
          % (r['name'], r['f1'], pc[0], pc[1], pc[2], pc[3], r['minutes']))

In [ ]:
impD = rD['imp']; order = np.argsort(impD)[::-1]
print('=== Model D feature importance (gain, fold 평균) ===')
for j in order:
    print('  %-14s %12.1f' % (FEAT_D[j], impD[j]))

# SHAP summary (선택). shap 미설치/실패 시 건너뜀.
try:
    import shap
    samp = np.random.RandomState(SEED).choice(N, size=min(2000, N), replace=False)
    expl = shap.TreeExplainer(rD['model'])
    sv = expl.shap_values(X['D 7d'][samp])
    shap.summary_plot(sv, X['D 7d'][samp], feature_names=FEAT_D, show=True)
except Exception as e:
    print('SHAP 생략:', e)

## 최종 의사결정 · 앙상블 OOF 저장

- Δ_fusion = F1(B) − F1(A)  (융합 모델)
- Δ_aisonly = F1(D) − F1(C)  (AIS-only 모델)

In [ ]:
fa = agg(res_ac['A'])[0]; fb = agg(res_ac['B'])[0]
fC = rC['f1']; fD = rD['f1']; fDp = rDp['f1']
d_fusion = fb - fa
d_aisonly = fD - fC
print('F1(A)=%.4f  F1(B)=%.4f  -> Δ_fusion=%+.4f' % (fa, fb, d_fusion))
print('F1(C)=%.4f  F1(D)=%.4f  -> Δ_aisonly=%+.4f' % (fC, fD, d_aisonly))

if d_aisonly < 0.01:
    dec = 'HDG는 AIS 자체에서도 무의미 -> 영구 제외. 모델 A(4-d) 확정.'
elif d_aisonly >= 0.02 and d_fusion < 0.005:
    dec = 'HDG 신호는 있으나 음향이 이미 포착 -> 융합은 A, 앙상블 AIS-only는 D 사용.'
elif d_aisonly >= 0.02 and d_fusion >= 0.01:
    dec = 'HDG가 음향과 독립 -> 모델 B(7-d) 확정.'
else:
    dec = '경계 구간 -> fold 1,2 추가 실행 후 평균으로 재판단 권장.'
print('결정(융합):', dec)

dd = abs(fD - fDp)
if dd < 0.005:
    print('D vs Dp: 차이 %.4f -> 더 단순한 Dp(raw NaN) 권장' % dd)
elif fDp > fD:
    print('Dp(raw) 우세 -> AIS-only 보조는 Dp, 음향 융합 인코딩은 D 유지(MLP는 NaN 불가)')
else:
    print('D 우세 -> D 인코딩 유지')

# ---- 앙상블용 OOF 저장 (나중에 가중평균 그리드서치) ----
chosen_ac = oof_B if d_fusion >= 0.01 else oof_A
cov = int((~np.isnan(chosen_ac[:, 0])).sum())
np.save(OOF_DIR + '/oof_acoustic_fusion.npy', chosen_ac)
ais_chosen = rDp['oof'] if (abs(fD - fDp) < 0.005 or fDp > fD) else rD['oof']
np.save(OOF_DIR + '/oof_ais_lgbm.npy', ais_chosen)
print('저장 완료 ->', OOF_DIR)
print('oof_acoustic_fusion.npy 채움: %d/%d  (fold %s만 실행. 전체 OOF는 FOLDS_ACOUSTIC=list(range(5)) 후 재실행)' % (cov, N, FOLDS_ACOUSTIC))
print('oof_ais_lgbm.npy 채움: %d/%d  (LGBM 5-fold 전체)' % (N, N))